# End-to-End Android APK Vulnerability Scanner Pipeline
This notebook automatically orchestrates `jadx` decompilation, AST parsing with `tree-sitter`, Data Flow Graph generation, and GraphCodeBERT inference natively in a Kaggle environment for an entire directory of APKs.

In [1]:
!pip install torch transformers tree_sitter==0.21.3 androguard==3.3.3 tqdm -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 659.7/659.7 kB 37.5 MB/s eta 0:00:00


In [2]:
import os
import shutil
import urllib.request
import zipfile
import subprocess
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from transformers import RobertaConfig, RobertaModel, AutoTokenizer
from tree_sitter import Language, Parser

# --- 1. DFG UTILS ---
def setup_tree_sitter():
    build_dir = 'build'
    lib_path = os.path.join(build_dir, 'my-languages.so')
    java_repo = 'tree-sitter-java'
    kotlin_repo = 'tree-sitter-kotlin'

    if not os.path.exists(lib_path):
        os.makedirs(build_dir, exist_ok=True)
        repos = []
        if not os.path.exists(java_repo):
            print(f"Cloning {java_repo}...")
            os.system(f"git clone https://github.com/tree-sitter/tree-sitter-java")
        repos.append(java_repo)
        
        if not os.path.exists(kotlin_repo):
            print(f"Cloning {kotlin_repo}...")
            os.system(f"git clone https://github.com/fwcd/tree-sitter-kotlin")
        repos.append(kotlin_repo)
        
        print("Compiling tree-sitter languages (Java + Kotlin)...")
        Language.build_library(lib_path, repos)
        print("Compilation complete.")

    try:
        java_lang = Language(lib_path, 'java')
        kotlin_lang = Language(lib_path, 'kotlin')
        parser = Parser()
        parser.set_language(java_lang)
        return (java_lang, kotlin_lang), parser
    except Exception as e:
        print(f"CRITICAL ERROR during parser setup: {e}")
        return None, None

def tree_to_token_index(root_node):
    if (len(root_node.children) == 0 or root_node.type == 'string') and root_node.type != 'comment':
        return [(root_node.start_point, root_node.end_point)]
    else:
        code_tokens = []
        for child in root_node.children:
            code_tokens += tree_to_token_index(child)
        return code_tokens

def tree_to_variable_index(root_node, index_to_code):
    if (len(root_node.children) == 0 or root_node.type == 'string') and root_node.type != 'comment':
        index = (root_node.start_point, root_node.end_point)
        _, code = index_to_code.get(index, (None, root_node.type))
        if root_node.type != code:
            return [(root_node.start_point, root_node.end_point)]
        else:
            return []
    else:
        code_tokens = []
        for child in root_node.children:
            code_tokens += tree_to_variable_index(child, index_to_code)
        return code_tokens

def index_to_code_token(index, code):
    start_point = index[0]
    end_point = index[1]
    if start_point[0] == end_point[0]:
        s = code[start_point[0]][start_point[1]:end_point[1]]
    else:
        s = ""
        s += code[start_point[0]][start_point[1]:]
        for i in range(start_point[0] + 1, end_point[0]):
            s += code[i]
        s += code[end_point[0]][:end_point[1]]
    return s

def DFG_java(root_node, index_to_code, states):
    assignment = ['assignment_expression']
    def_statement = ['variable_declarator']
    increment_statement = ['update_expression']
    if_statement = ['if_statement', 'else']
    for_statement = ['for_statement']
    enhanced_for_statement = ['enhanced_for_statement']
    while_statement = ['while_statement']
    do_first_statement = []
    
    states = states.copy()
    if (len(root_node.children) == 0 or root_node.type == 'string') and root_node.type != 'comment':
        idx, code = index_to_code.get((root_node.start_point, root_node.end_point), (0, root_node.type))
        if root_node.type == code:
            return [], states
        elif code in states:
            return [(code, idx, 'comesFrom', [code], states[code].copy())], states
        else:
            if root_node.type == 'identifier':
                states[code] = [idx]
            return [(code, idx, 'comesFrom', [], [])], states
            
    elif root_node.type in def_statement:
        name = root_node.child_by_field_name('name')
        value = root_node.child_by_field_name('value')
        DFG = []
        if value is None:
            indexs = tree_to_variable_index(name, index_to_code)
            for index in indexs:
                if index in index_to_code:
                    idx, code = index_to_code[index]
                    DFG.append((code, idx, 'comesFrom', [], []))
                    states[code] = [idx]
            return sorted(DFG, key=lambda x: x[1]), states
        else:
            name_indexs = tree_to_variable_index(name, index_to_code)
            value_indexs = tree_to_variable_index(value, index_to_code)
            temp, states = DFG_java(value, index_to_code, states)
            DFG += temp
            for index1 in name_indexs:
                if index1 in index_to_code:
                    idx1, code1 = index_to_code[index1]
                    for index2 in value_indexs:
                        if index2 in index_to_code:
                            idx2, code2 = index_to_code[index2]
                            DFG.append((code1, idx1, 'comesFrom', [code2], [idx2]))
                    states[code1] = [idx1]
            return sorted(DFG, key=lambda x: x[1]), states
            
    elif root_node.type in assignment:
        left_nodes = root_node.child_by_field_name('left')
        right_nodes = root_node.child_by_field_name('right')
        DFG = []
        temp, states = DFG_java(right_nodes, index_to_code, states)
        DFG += temp
        name_indexs = tree_to_variable_index(left_nodes, index_to_code)
        value_indexs = tree_to_variable_index(right_nodes, index_to_code)
        for index1 in name_indexs:
            if index1 in index_to_code:
                idx1, code1 = index_to_code[index1]
                for index2 in value_indexs:
                    if index2 in index_to_code:
                        idx2, code2 = index_to_code[index2]
                        DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
                states[code1] = [idx1]
        return sorted(DFG, key=lambda x: x[1]), states
        
    elif root_node.type in increment_statement:
        DFG = []
        indexs = tree_to_variable_index(root_node, index_to_code)
        for index1 in indexs:
            if index1 in index_to_code:
                idx1, code1 = index_to_code[index1]
                for index2 in indexs:
                    if index2 in index_to_code:
                        idx2, code2 = index_to_code[index2]
                        DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
                states[code1] = [idx1]
        return sorted(DFG, key=lambda x: x[1]), states
        
    elif root_node.type in if_statement:
        DFG = []
        current_states = states.copy()
        others_states = []
        flag = False
        tag = False
        if 'else' in root_node.type:
            tag = True
        for child in root_node.children:
            if 'else' in child.type:
                tag = True
            if child.type not in if_statement and flag is False:
                temp, current_states = DFG_java(child, index_to_code, current_states)
                DFG += temp
            else:
                flag = True
                temp, new_states = DFG_java(child, index_to_code, states)
                DFG += temp
                others_states.append(new_states)
        others_states.append(current_states)
        if tag is False:
            others_states.append(states)
        new_states = {}
        for dic in others_states:
            for key in dic:
                if key not in new_states:
                    new_states[key] = dic[key].copy()
                else:
                    new_states[key] += dic[key]
        for key in new_states:
            new_states[key] = sorted(list(set(new_states[key])))
        return sorted(DFG, key=lambda x: x[1]), new_states
        
    elif root_node.type in for_statement:
        DFG = []
        for child in root_node.children:
            temp, states = DFG_java(child, index_to_code, states)
            DFG += temp
        flag = False
        for child in root_node.children:
            if flag:
                temp, states = DFG_java(child, index_to_code, states)
                DFG += temp
            elif child.type == "local_variable_declaration":
                flag = True
        dic = {}
        for x in DFG:
            if (x[0], x[1], x[2]) not in dic:
                dic[(x[0], x[1], x[2])] = [x[3], x[4]]
            else:
                dic[(x[0], x[1], x[2])][0] = list(set(dic[(x[0], x[1], x[2])][0] + x[3]))
                dic[(x[0], x[1], x[2])][1] = sorted(list(set(dic[(x[0], x[1], x[2])][1] + x[4])))
        DFG = [(x[0], x[1], x[2], y[0], y[1]) for x, y in sorted(dic.items(), key=lambda t: t[0][1])]
        return sorted(DFG, key=lambda x: x[1]), states
        
    elif root_node.type in enhanced_for_statement:
        name = root_node.child_by_field_name('name')
        value = root_node.child_by_field_name('value')
        body = root_node.child_by_field_name('body')
        DFG = []
        for i in range(2):
            temp, states = DFG_java(value, index_to_code, states)
            DFG += temp
            name_indexs = tree_to_variable_index(name, index_to_code)
            value_indexs = tree_to_variable_index(value, index_to_code)
            for index1 in name_indexs:
                if index1 in index_to_code:
                    idx1, code1 = index_to_code[index1]
                    for index2 in value_indexs:
                        if index2 in index_to_code:
                            idx2, code2 = index_to_code[index2]
                            DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
                    states[code1] = [idx1]
            temp, states = DFG_java(body, index_to_code, states)
            DFG += temp
        dic = {}
        for x in DFG:
            if (x[0], x[1], x[2]) not in dic:
                dic[(x[0], x[1], x[2])] = [x[3], x[4]]
            else:
                dic[(x[0], x[1], x[2])][0] = list(set(dic[(x[0], x[1], x[2])][0] + x[3]))
                dic[(x[0], x[1], x[2])][1] = sorted(list(set(dic[(x[0], x[1], x[2])][1] + x[4])))
        DFG = [(x[0], x[1], x[2], y[0], y[1]) for x, y in sorted(dic.items(), key=lambda t: t[0][1])]
        return sorted(DFG, key=lambda x: x[1]), states
        
    elif root_node.type in while_statement:
        DFG = []
        for i in range(2):
            for child in root_node.children:
                temp, states = DFG_java(child, index_to_code, states)
                DFG += temp
        dic = {}
        for x in DFG:
            if (x[0], x[1], x[2]) not in dic:
                dic[(x[0], x[1], x[2])] = [x[3], x[4]]
            else:
                dic[(x[0], x[1], x[2])][0] = list(set(dic[(x[0], x[1], x[2])][0] + x[3]))
                dic[(x[0], x[1], x[2])][1] = sorted(list(set(dic[(x[0], x[1], x[2])][1] + x[4])))
        DFG = [(x[0], x[1], x[2], y[0], y[1]) for x, y in sorted(dic.items(), key=lambda t: t[0][1])]
        return sorted(DFG, key=lambda x: x[1]), states
        
    else:
        DFG = []
        for child in root_node.children:
            if child.type in do_first_statement:
                temp, states = DFG_java(child, index_to_code, states)
                DFG += temp
        for child in root_node.children:
            if child.type not in do_first_statement:
                temp, states = DFG_java(child, index_to_code, states)
                DFG += temp
        return sorted(DFG, key=lambda x: x[1]), states

def DFG_kotlin(root_node, index_to_code, states):
    assignment = ['assignment']
    def_statement = ['property_declaration', 'variable_declaration', 'parameter']
    increment_statement = ['postfix_expression', 'prefix_expression']
    if_statement = ['if_expression', 'else']
    for_statement = ['for_statement']
    while_statement = ['while_statement', 'do_while_statement']
    
    states = states.copy()
    if (len(root_node.children) == 0 or root_node.type == 'string') and root_node.type != 'comment':
        idx, code = index_to_code.get((root_node.start_point, root_node.end_point), (0, root_node.type))
        if root_node.type == code:
            return [], states
        elif code in states:
            return [(code, idx, 'comesFrom', [code], states[code].copy())], states
        else:
            if root_node.type == 'simple_identifier':
                states[code] = [idx]
            return [(code, idx, 'comesFrom', [], [])], states
            
    elif root_node.type in def_statement:
        name_node = None
        value_node = None
        for child in root_node.children:
            if child.type == 'variable_declaration' or child.type == 'simple_identifier':
                name_node = child
            if child.type in ['expression', 'call_expression', 'literal_constant']:
                value_node = child
        
        if name_node is None:
            return [], states
        DFG = []
        if value_node is None:
            indexs = tree_to_variable_index(name_node, index_to_code)
            for index in indexs:
                if index in index_to_code:
                    idx, code = index_to_code[index]
                    DFG.append((code, idx, 'comesFrom', [], []))
                    states[code] = [idx]
            return sorted(DFG, key=lambda x: x[1]), states
        else:
            name_indexs = tree_to_variable_index(name_node, index_to_code)
            value_indexs = tree_to_variable_index(value_node, index_to_code)
            temp, states = DFG_kotlin(value_node, index_to_code, states)
            DFG += temp
            for index1 in name_indexs:
                if index1 in index_to_code:
                    idx1, code1 = index_to_code[index1]
                    for index2 in value_indexs:
                        if index2 in index_to_code:
                            idx2, code2 = index_to_code[index2]
                            DFG.append((code1, idx1, 'comesFrom', [code2], [idx2]))
                    states[code1] = [idx1]
            return sorted(DFG, key=lambda x: x[1]), states

    elif root_node.type in assignment:
        left_node = root_node.children[0]
        right_node = root_node.children[2] if len(root_node.children) > 2 else None
        DFG = []
        if right_node:
            temp, states = DFG_kotlin(right_node, index_to_code, states)
            DFG += temp
            name_indexs = tree_to_variable_index(left_node, index_to_code)
            value_indexs = tree_to_variable_index(right_node, index_to_code)
            for index1 in name_indexs:
                if index1 in index_to_code:
                    idx1, code1 = index_to_code[index1]
                    for index2 in value_indexs:
                        if index2 in index_to_code:
                            idx2, code2 = index_to_code[index2]
                            DFG.append((code1, idx1, 'computedFrom', [code2], [idx2]))
                    states[code1] = [idx1]
        return sorted(DFG, key=lambda x: x[1]), states

    else:
        DFG = []
        for child in root_node.children:
            temp, states = DFG_kotlin(child, index_to_code, states)
            DFG += temp
        return sorted(DFG, key=lambda x: x[1]), states

def clean_and_wrap_code(code_snippet, lang='java'):
    code_snippet = code_snippet.strip()
    if lang == 'kotlin':
        return code_snippet
    if "class " in code_snippet and "{" in code_snippet:
        return code_snippet
    if any(x in code_snippet for x in ["public ", "private ", "protected "]):
        return f"public class DummyClass {{ {code_snippet} }}"
    return f"public class DummyClass {{ public void dummyMethod() {{ {code_snippet} }} }}"

def extract_dfg_from_code(code_snippet, parser, lang='java'):
    wrapped_code = clean_and_wrap_code(code_snippet, lang)
    tree = parser.parse(bytes(wrapped_code, 'utf8'))
    root_node = tree.root_node
    
    tokens_index = tree_to_token_index(root_node)
    code_lines = wrapped_code.split('\n')
    index_to_code = {}
    
    try:
        for idx, (start, end) in enumerate(tokens_index):
            string_val = index_to_code_token((start, end), code_lines)
            index_to_code[(start, end)] = ((start, end), string_val)
            
        if lang == 'kotlin':
            DFG, _ = DFG_kotlin(root_node, index_to_code, {})
        else:
            DFG, _ = DFG_java(root_node, index_to_code, {})
        return DFG
    except Exception as e:
        return []

def extract_methods_from_file(file_path, parser, langs):
    java_lang, kotlin_lang = langs
    is_kotlin = file_path.endswith('.kt')
    
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            code = f.read()
    except Exception as e:
        return []

    if is_kotlin:
        parser.set_language(kotlin_lang)
        method_types = ['function_declaration', 'secondary_constructor', 'primary_constructor']
    else:
        parser.set_language(java_lang)
        method_types = ['method_declaration', 'constructor_declaration']

    tree = parser.parse(bytes(code, 'utf8'))
    methods = []
    
    def traverse(node):
        if node.type in method_types:
            start_byte = node.start_byte
            end_byte = node.end_byte
            method_code = code[start_byte:end_byte]
            if len(method_code.splitlines()) > 2 and ("{" in method_code or "=" in method_code):
                methods.append({
                    "file": file_path,
                    "code": method_code,
                    "lang": 'kotlin' if is_kotlin else 'java'
                })
        for child in node.children:
            traverse(child)
            
    traverse(tree.root_node)
    return methods


In [3]:
# --- 2. MODEL UTILS (UniXcoder text-only) ---

class SimpleModel(nn.Module):
    def __init__(self, encoder, config):
        super(SimpleModel, self).__init__()
        self.encoder = encoder
        self.config = config
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0]  # [Batch, Seq, Hidden]
        # Use CLS token for classification
        logits = self.classifier(self.dropout(sequence_output[:, 0, :]))
        prob = F.softmax(logits, dim=-1)
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            return loss, prob
        return prob


def load_model(weights_path, device):
    print(f"Loading UniXcoder (text-only) from {weights_path}...")
    cfg = RobertaConfig.from_pretrained("microsoft/graphcodebert-base")
    cfg.num_labels = 2
    tok = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    enc = RobertaModel.from_pretrained("microsoft/graphcodebert-base", config=cfg)

    model = SimpleModel(enc, cfg).to(device)
    try:
        model.load_state_dict(torch.load(weights_path, map_location=device))
        model.eval()
        print("  ✓ Model loaded successfully")
        return model, tok
    except Exception as e:
        print(f"Error loading model weights: {e}")
        return None, None


def infer_vulnerability(code, tokenizer, model, device, threshold=0.45, code_length=384):
    """Run inference on a single code snippet (text-only, no DFG)."""
    tokens = tokenizer(
        code,
        max_length=code_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    input_ids = tokens["input_ids"].to(device)
    attention_mask = tokens["attention_mask"].to(device)

    with torch.no_grad():
        prob = model(input_ids=input_ids, attention_mask=attention_mask)

    p_vuln = prob[0][1].item()
    return p_vuln >= threshold, p_vuln


def infer_vulnerabilities_batched(methods, model, tokenizer, device,
                                   threshold=0.45, code_length=384, batch_size=32):
    """Batch inference for a list of (m_id, code, dfg_unused, short_class) tuples.

    The dfg field is accepted but ignored – this model is text-only.
    """
    results = {}
    class_map = {}

    try:
        from tqdm import tqdm
        chunk_iterable = tqdm(range(0, len(methods), batch_size),
                              desc="Inferring in GPU batches", leave=True)
    except ImportError:
        chunk_iterable = range(0, len(methods), batch_size)

    for i in chunk_iterable:
        batch = methods[i: i + batch_size]
        codes = [m[1] for m in batch]

        tokens = tokenizer(
            codes,
            max_length=code_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = tokens["input_ids"].to(device)
        attention_mask = tokens["attention_mask"].to(device)

        with torch.no_grad():
            probs = model(input_ids=input_ids, attention_mask=attention_mask)

        for idx, chunk_prob in enumerate(probs):
            m_id = batch[idx][0]
            p_vuln = chunk_prob[1].item()
            class_map[m_id] = batch[idx][3]
            results[m_id] = max(results.get(m_id, 0.0), p_vuln)

    final_results = [
        (m_id, results[m_id] >= threshold, results[m_id], class_map[m_id])
        for m_id in results
    ]
    return final_results


In [4]:
# --- 3. EXECUTION ---
JADX_VERSION = "1.4.7"
JADX_URL = f"https://github.com/skylot/jadx/releases/download/v{JADX_VERSION}/jadx-{JADX_VERSION}.zip"
TOOLS_DIR = Path("/kaggle/working/tools")
JADX_BIN = TOOLS_DIR / "jadx" / "bin" / "jadx"
OUT_DIR = Path("/kaggle/working/out")

def check_and_install_jadx():
    if shutil.which("jadx"):
        print("[*] JADX found in system PATH.")
        return "jadx"
    
    if JADX_BIN.exists():
        print(f"[*] Local JADX found at {JADX_BIN}")
        return str(JADX_BIN)
        
    print("[!] JADX not found. Downloading...")
    TOOLS_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = TOOLS_DIR / "jadx.zip"
    
    urllib.request.urlretrieve(JADX_URL, zip_path)
    print("[*] Extracting JADX...")
    jadx_dir = TOOLS_DIR / "jadx"
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(jadx_dir)
        
    os.chmod(JADX_BIN, 0o755)
    zip_path.unlink()
    print("[*] JADX installed successfully.")
    return str(JADX_BIN)

def decompile_apk(jadx_path, apk_path, output_dir):
    print(f"[*] Decompiling {apk_path} to {output_dir}")
    os.makedirs(output_dir, exist_ok=True)
    cmd = [jadx_path, "--no-res", "-d", str(output_dir), str(apk_path)]
    try:
        subprocess.run(cmd, check=True, timeout=600, capture_output=True, text=True)
        print("[*] Decompilation finished.")
        return True
    except subprocess.TimeoutExpired:
        print("[!] Decompilation timed out.")
        return False
    except subprocess.CalledProcessError as e:
        print(f"[!] Decompilation failed: {e.stderr}")
        return False

def scan_directory_for_code(src_dir, target_package=None):
    code_files = []
    package_path = target_package.replace('.', os.sep) if target_package else None
    
    for root, _, files in os.walk(src_dir):
        if package_path and package_path not in root:
            continue
        for file in files:
            if file.endswith(".java") or file.endswith(".kt"):
                code_files.append(os.path.join(root, file))
    return code_files

# =========================================================================
# KAGGLE PATH CONFIGURATION
# =========================================================================
APK_DIR = Path("/kaggle/input/datasets/hasanmahmudabdullah/testapks/APKs/")
MODEL_PATH = Path("/kaggle/input/notebooks/iahmed223141/graphcodebert-train-text-only/saved_models/best_model_text_only.bin")  # GCB text-only weights
THRESHOLD = 0.45
TARGET_PACKAGE = "auto"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Using device: {device}")

# Run pipeline
if not APK_DIR.exists():
    print(f"[!] Error: Could not find APK directory at {APK_DIR}.")
elif not MODEL_PATH.exists():
    print(f"[!] Error: Could not find model at {MODEL_PATH}.")
else:
    apk_files = list(APK_DIR.glob("*.apk"))
    print(f"[*] Found {len(apk_files)} APKs in {APK_DIR}")
    
    jadx_cmd = check_and_install_jadx()
    
    print("[*] Setting up Tree-sitter parsers...")
    langs, ts_parser = setup_tree_sitter()
    
    master_summary = []
    
    if ts_parser:
        model, tokenizer = load_model(MODEL_PATH, device)
        if model:
            for apk_path in apk_files:
                print(f"\n{'='*60}")
                print(f"[*] Analyzing: {apk_path.name}")
                print(f"{'='*60}")
                
                target_out_dir = OUT_DIR / apk_path.stem
                report_path = OUT_DIR / f"{apk_path.stem}_vuln_report.json"
                
                if report_path.exists():
                    print(f"[*] Skipping {apk_path.name}, report already exists.")
                    continue
                
                active_target_pkg = TARGET_PACKAGE
                if active_target_pkg == "auto":
                    try:
                        from androguard.core.bytecodes.apk import APK
                        a = APK(str(apk_path))
                        active_target_pkg = a.get_package()
                        print(f"[*] Auto-detected Target Package: {active_target_pkg}")
                    except:
                        active_target_pkg = None
                
                if decompile_apk(jadx_cmd, apk_path, target_out_dir):
                    code_files = scan_directory_for_code(target_out_dir, active_target_pkg)
                    print(f"[*] Found {len(code_files)} source files (Java/Kotlin) to analyze.")

                    all_methods = []
                    m_id_counter = 0

                    try:
                        from tqdm import tqdm
                        file_iterable = tqdm(code_files, desc="Parsing AST & DFG", leave=True)
                    except:
                        file_iterable = code_files

                    for f_path in file_iterable:
                        methods = extract_methods_from_file(f_path, ts_parser, langs)
                        for m in methods:
                            short_class = Path(f_path).stem
                            # DFG is not used by the text-only model; pass empty list for API compatibility
                            all_methods.append((m_id_counter, m['code'], [], short_class))
                            m_id_counter += 1
                            
                    print(f"[*] Extracted {len(all_methods)} Java/Kotlin methods for Inference.")

                    report_data = {
                        "apk_name": apk_path.name,
                        "total_functions_scanned": len(all_methods),
                        "vulnerable_functions": [],
                        "all_probabilities": [],
                        "vulnerable_functions_count": 0,
                        "safe_functions_count": 0
                    }

                    if len(all_methods) > 0:
                        results = infer_vulnerabilities_batched(all_methods, model, tokenizer, device, THRESHOLD)
                        total_vuln = 0
                        for m_id, is_vuln, prob, short_class in results:
                            report_data["all_probabilities"].append(round(prob, 4))
                            if is_vuln:
                                total_vuln += 1
                                report_data["vulnerable_functions"].append({"class": short_class, "prob": round(prob, 4)})
                        
                        report_data["vulnerable_functions_count"] = total_vuln
                        report_data["safe_functions_count"] = len(all_methods) - total_vuln
                        
                        print("-" * 50)
                        print(f"Summary for {apk_path.name}")
                        print(f"  Total Functions Scanned : {len(all_methods)}")
                        print(f"  Vulnerable Functions    : {total_vuln}")
                        print("-" * 50)

                    with open(report_path, "w") as f:
                        import json
                        json.dump(report_data, f, indent=4)
                    
                    master_summary.append({
                        "apk_name": apk_path.name,
                        "total": len(all_methods),
                        "vuln": report_data["vulnerable_functions_count"]
                    })
            
            if master_summary:
                import csv
                csv_path = OUT_DIR / "master_summary.csv"
                with open(csv_path, "w", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=["apk_name", "total", "vuln"])
                    writer.writeheader()
                    writer.writerows(master_summary)
                print(f"\n[*] Master CSV summary saved to: {csv_path}")


[*] Using device: cuda
[*] Found 13 APKs in /kaggle/input/datasets/hasanmahmudabdullah/testapks/APKs
[!] JADX not found. Downloading...
[*] Extracting JADX...
[*] JADX installed successfully.
[*] Setting up Tree-sitter parsers...
Cloning tree-sitter-java...


Cloning into 'tree-sitter-java'...


Cloning tree-sitter-kotlin...


Cloning into 'tree-sitter-kotlin'...


Compiling tree-sitter languages (Java + Kotlin)...


...

--------------------------------------------------
Summary for de.danoeh.antennapod_3110095.apk
  Total Functions Scanned : 6169
  Vulnerable Functions    : 621
--------------------------------------------------

[*] Analyzing: InsecureShop.apk
[*] Auto-detected Target Package: com.insecureshop
[*] Decompiling /kaggle/input/datasets/hasanmahmudabdullah/testapks/APKs/InsecureShop.apk to /kaggle/working/out/InsecureShop
[*] Decompilation finished.
[*] Found 51 source files (Java/Kotlin) to analyze.


Parsing AST & DFG: 100%|██████████| 51/51 [00:00<00:00, 134.73it/s]


[*] Extracted 336 Java/Kotlin methods for Inference.


Inferring in GPU batches: 100%|██████████| 11/11 [00:07<00:00,  1.50it/s]

--------------------------------------------------
Summary for InsecureShop.apk
  Total Functions Scanned : 336
  Vulnerable Functions    : 14
--------------------------------------------------

[*] Master CSV summary saved to: /kaggle/working/out/master_summary.csv
